# Day 075 — Solution: Talking-Head Pipeline

In [ ]:
_SRC = '"""talking_head.py — Day 075: Talking-Head Video Pipeline.\n\nCreate a talking-head video from a text script:\n1. generate_speech  — TTS audio bytes (edge-tts)\n2. image_to_frames  — face image → repeated BGR numpy frame list\n3. mux_audio_video  — FFmpeg: combine silent video + audio → MP4\n4. add_captions     — FFmpeg drawtext: burn caption onto video\n\nAll functions accept injection callables for headless testing.\n\nJupyter note: generate_speech uses asyncio.run(). In Jupyter, first run:\n    import nest_asyncio; nest_asyncio.apply()\n\nSetup:\n    pip install edge-tts opencv-python-headless Pillow\n    brew install ffmpeg   # macOS\n"""\nimport subprocess\nimport tempfile\nfrom pathlib import Path\nfrom typing import Callable, Optional\n\n\ndef generate_speech(text: str, voice: str = \'en-US-AriaNeural\',\n                    rate: str = \'+0%\', pitch: str = \'+0Hz\',\n                    tts_fn: Optional[Callable] = None) -> bytes:\n    """Generate speech audio bytes from text using edge-tts.\n\n    Args:\n        text:   text to synthesise\n        voice:  edge-tts ShortName (e.g. \'en-US-AriaNeural\')\n        rate:   speaking rate adjustment (\'+10%\', \'-5%\', ...)\n        pitch:  pitch adjustment (\'+5Hz\', \'-10Hz\', ...)\n        tts_fn: callable(text, voice, rate, pitch) -> bytes for testing\n    Returns:\n        MP3 audio bytes\n    """\n    if tts_fn is not None:\n        return tts_fn(text, voice, rate, pitch)\n    import asyncio\n    import edge_tts\n\n    async def _run():\n        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)\n        chunks = []\n        async for chunk in comm.stream():\n            if chunk[\'type\'] == \'audio\':\n                chunks.append(chunk[\'data\'])\n        return b\'\'.join(chunks)\n\n    return asyncio.run(_run())\n\n\ndef image_to_frames(image, n_frames: int,\n                    capture_fn: Optional[Callable] = None) -> list:\n    """Create n_frames identical BGR frames from a face image.\n\n    Args:\n        image:      PIL Image object or path to image file (str or Path)\n        n_frames:   number of frames to generate\n        capture_fn: callable(image, n_frames) -> list[np.ndarray] for testing\n    Returns:\n        list of numpy arrays (H, W, 3) uint8 BGR — n_frames elements\n    """\n    if capture_fn is not None:\n        return capture_fn(image, n_frames)\n    import numpy as np\n    from PIL import Image as PILImage\n    if not isinstance(image, PILImage.Image):\n        image = PILImage.open(str(image)).convert(\'RGB\')\n    arr = np.array(image)[:, :, ::-1].astype(np.uint8)  # RGB -> BGR\n    return [arr.copy() for _ in range(n_frames)]\n\n\ndef mux_audio_video(video_path, audio_bytes: bytes, output_path,\n                    ffmpeg_fn: Optional[Callable] = None) -> Path:\n    """Combine a silent video file with audio bytes into a new MP4.\n\n    Args:\n        video_path:  path to silent video file\n        audio_bytes: MP3 audio bytes to add as soundtrack\n        output_path: destination MP4 path\n        ffmpeg_fn:   callable(video_path, audio_bytes, output_path) -> Path\n    Returns:\n        Path to the output video with audio\n    """\n    if ffmpeg_fn is not None:\n        return ffmpeg_fn(video_path, audio_bytes, output_path)\n    with tempfile.NamedTemporaryFile(suffix=\'.mp3\', delete=False) as f:\n        f.write(audio_bytes)\n        audio_path = f.name\n    try:\n        out_path = Path(output_path)\n        result = subprocess.run(\n            [\'ffmpeg\', \'-y\', \'-i\', str(video_path), \'-i\', audio_path,\n             \'-c:v\', \'copy\', \'-c:a\', \'aac\', \'-shortest\', str(out_path)],\n            capture_output=True, text=True,\n        )\n        if result.returncode != 0:\n            raise RuntimeError(f\'FFmpeg mux error: {result.stderr[-500:]}\')\n        return out_path\n    finally:\n        Path(audio_path).unlink(missing_ok=True)\n\n\ndef add_captions(video_path, text: str, output_path,\n                 fontsize: int = 24, color: str = \'white\',\n                 ffmpeg_fn: Optional[Callable] = None) -> Path:\n    """Burn a caption onto a video using FFmpeg drawtext filter.\n\n    Args:\n        video_path:  source video\n        text:        caption text (single line)\n        output_path: destination path\n        fontsize:    font size in pixels\n        color:       text colour (\'white\', \'yellow\', \'black\', ...)\n        ffmpeg_fn:   callable(video_path, text, output_path) -> Path\n    Returns:\n        Path to captioned video\n    """\n    if ffmpeg_fn is not None:\n        return ffmpeg_fn(video_path, text, output_path)\n    safe_text = text.replace("\'", r"\\\'").replace(\':\', r\'\\:\')\n    out_path = Path(output_path)\n    result = subprocess.run(\n        [\'ffmpeg\', \'-y\', \'-i\', str(video_path),\n         \'-vf\', (f"drawtext=text=\'{safe_text}\':fontsize={fontsize}:"\n                 f"fontcolor={color}:x=(w-text_w)/2:y=h-text_h-20"),\n         str(out_path)],\n        capture_output=True, text=True,\n    )\n    if result.returncode != 0:\n        raise RuntimeError(f\'FFmpeg caption error: {result.stderr[-500:]}\')\n    return out_path\n\n\nclass TalkingHeadPipeline:\n    """Create talking-head videos from text and a face image.\n\n    Bind injection functions at construction for easy testing::\n\n        pipe = TalkingHeadPipeline(\n            tts_fn=lambda text, voice, rate, pitch: b\'MP3_AUDIO\',\n            capture_fn=lambda img, n: [np.zeros((64,64,3), np.uint8)] * n,\n            mux_fn=lambda vp, ab, op: (Path(op).write_bytes(b\'V\'), Path(op))[1],\n            caption_fn=lambda vp, t, op: (Path(op).write_bytes(b\'C\'), Path(op))[1],\n        )\n    """\n\n    def __init__(self, tts_fn: Optional[Callable] = None,\n                 capture_fn: Optional[Callable] = None,\n                 mux_fn: Optional[Callable] = None,\n                 caption_fn: Optional[Callable] = None) -> None:\n        self._tts_fn     = tts_fn\n        self._capture_fn = capture_fn\n        self._mux_fn     = mux_fn\n        self._caption_fn = caption_fn\n\n    def speech(self, text: str, voice: str = \'en-US-AriaNeural\',\n               rate: str = \'+0%\', pitch: str = \'+0Hz\') -> bytes:\n        """Generate speech audio bytes."""\n        return generate_speech(text, voice=voice, rate=rate, pitch=pitch,\n                               tts_fn=self._tts_fn)\n\n    def frames(self, image, n_frames: int) -> list:\n        """Return n_frames BGR numpy arrays from a face image."""\n        return image_to_frames(image, n_frames, capture_fn=self._capture_fn)\n\n    def mux(self, video_path, audio_bytes: bytes, output_path) -> Path:\n        """Combine silent video + audio into output MP4."""\n        return mux_audio_video(video_path, audio_bytes, output_path,\n                               ffmpeg_fn=self._mux_fn)\n\n    def caption(self, video_path, text: str, output_path,\n                fontsize: int = 24, color: str = \'white\') -> Path:\n        """Burn a caption onto a video."""\n        return add_captions(video_path, text, output_path,\n                            fontsize=fontsize, color=color,\n                            ffmpeg_fn=self._caption_fn)\n'
from pathlib import Path
Path('talking_head.py').write_text(_SRC, encoding='utf-8')
print('talking_head.py written.')

In [ ]:
import tempfile
from pathlib import Path
from talking_head import (
    generate_speech, image_to_frames, mux_audio_video,
    add_captions, TalkingHeadPipeline,
)

def _make_mock_frames(n=5, height=32, width=32):
    import numpy as np
    return [np.zeros((height, width, 3), dtype=np.uint8) for _ in range(n)]

_mock_tts     = lambda text, voice, rate, pitch: b'MP3:' + text[:8].encode()
_mock_capture = lambda img, n: _make_mock_frames(n)
_mock_mux     = lambda vp, ab, op: (Path(op).write_bytes(b'MUX' + bytes(len(ab))), Path(op))[1]
_mock_caption = lambda vp, t, op: (Path(op).write_bytes(b'CAP' + bytes(len(t))), Path(op))[1]

# 1. generate_speech
audio = generate_speech('Hello Day 75!', tts_fn=_mock_tts)
assert isinstance(audio, bytes) and len(audio) > 0
print("\u2705 generate_speech correct")

# 2. image_to_frames
frames = image_to_frames('face.png', 5, capture_fn=_mock_capture)
assert len(frames) == 5 and frames[0].shape == (32, 32, 3)
print("\u2705 image_to_frames correct")

# 3. mux_audio_video
with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f: silent = f.name
Path(silent).write_bytes(b'SILENT')
with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f: muxed = f.name
m = mux_audio_video(silent, audio, muxed, ffmpeg_fn=_mock_mux)
assert isinstance(m, Path) and m.exists() and m.stat().st_size > 0
print("\u2705 mux_audio_video correct")

# 4. add_captions
with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f: captioned = f.name
c = add_captions(silent, 'Day 75', captioned, ffmpeg_fn=_mock_caption)
assert isinstance(c, Path) and c.exists()
print("\u2705 add_captions correct")

# 5. TalkingHeadPipeline
pipe = TalkingHeadPipeline(tts_fn=_mock_tts, capture_fn=_mock_capture,
                           mux_fn=_mock_mux, caption_fn=_mock_caption)
a = pipe.speech('Section 5 pipeline complete!')
f2 = pipe.frames('face.png', 10)
with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f: s2 = f.name
Path(s2).write_bytes(b'SILENT')
with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f: m2 = f.name
with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f: c2 = f.name
mx = pipe.mux(s2, a, m2)
cp = pipe.caption(mx, 'Day 75 done!', c2)
assert isinstance(a, bytes) and len(f2) == 10
assert isinstance(mx, Path) and mx.exists()
assert isinstance(cp, Path) and cp.exists()
print("\u2705 TalkingHeadPipeline correct")
print("\nTalking-Head Pipeline complete!")
